# 9f — Temporal Spread per Cluster

**Goal:** Compute publication year spread per cluster.  
- Tight IQR → emerging niche | Wide IQR → established field

**2×2 quadrant:** rocket / niche / classic / stable

**Output:** `temporal_cohesion.json` + merges `temporalCohesion` field into `clusterexploration.json`

**Prerequisites:** `cluster_labels_500d.pkl`, `arxiv_metadata_features.pkl`

In [ ]:
import pickle, json, os
import numpy as np
import pandas as pd

DATA = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
OUT  = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'arxiv-trends-website', 'src', 'data')

with open(os.path.join(DATA, 'cluster_labels_500d.pkl'), 'rb') as f:
    labels = pickle.load(f)
with open(os.path.join(DATA, 'arxiv_metadata_features.pkl'), 'rb') as f:
    meta = pickle.load(f)

df = pd.DataFrame({'cluster': labels, 'year': meta['year']})
df = df[df['year'].between(2007, 2025)]
print(f'Papers with valid year: {len(df):,}')

In [ ]:
RECENT_CUTOFF = 2020

results = []
for cid in sorted(df['cluster'].unique()):
    years = df[df['cluster'] == cid]['year'].values
    q1, q3 = float(np.percentile(years, 25)), float(np.percentile(years, 75))
    iqr = q3 - q1
    emergence_score = float((years >= RECENT_CUTOFF).mean())
    results.append({
        'clusterId': int(cid),
        'meanYear':  round(float(np.mean(years)), 1),
        'medianYear': round(float(np.median(years)), 1),
        'stdYear':   round(float(np.std(years)), 2),
        'iqr':       round(iqr, 2),
        'q1Year':    round(q1, 1),
        'q3Year':    round(q3, 1),
        'emergenceScore': round(emergence_score, 4),
        'tight': iqr <= 4,
        'recent': emergence_score >= 0.5
    })

tc_df = pd.DataFrame(results)
print(tc_df.sort_values('emergenceScore', ascending=False)[['clusterId','medianYear','iqr','emergenceScore']].head(15).to_string())

# Save
out_path = os.path.join(OUT, 'temporal_cohesion.json')
with open(out_path, 'w') as f:
    json.dump({'clusters': results}, f, indent=2)
print(f'Saved → {out_path}')

# Merge into clusterexploration.json
ce_path = os.path.join(OUT, 'clusterexploration.json')
with open(ce_path) as f:
    ce = json.load(f)
tc_by_id = {r['clusterId']: r for r in results}
for cluster in ce['clusters']:
    if cluster['id'] in tc_by_id:
        cluster['temporalCohesion'] = tc_by_id[cluster['id']]
with open(ce_path, 'w') as f:
    json.dump(ce, f, indent=2)
print('Merged into clusterexploration.json')